# Rare-word sparse retrieval — diagnostics

| Section | What runs | Gate |
|---|---|---|
| Step 1 | `rarity` — the frozen 500-word list | the head of the list is vocabulary, not tokenizer debris |
| Step 2 | `leakage` — near-duplicate audit, quarantine | few enough flags that quarantining leaves the pool intact |
| Step 3 | `sparse_select` — selection dry run | full dose ≥ 30% of val queries, else report and stop |

The method: count how many training sentences carry each source word, freeze the 500
rarest, and for each val query take up to the four rarest listed words it carries. Each
selected word contributes one exemplar — the training example containing it that is
nearest the query — and ordinary cosine kNN fills the rest of the eight prompt slots.

The df floor is the one parameter that decides whether the channel fires. The 500 rarest
words in the whole vocabulary all sit at df 1 and no val query carries them; the floor
raises the list to words rare enough to be marked and common enough to recur.

In [ ]:
# e5-large in fp32 over a 10.8k-row pool; any Colab GPU is enough.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /home/prnamhr/projects/Style-Aware-MT
!pip install -r requirements.txt

# Text-only pipeline; these two carry an ABI mismatch against the pinned torch.
!pip uninstall -y torchvision torchaudio

`data/knn_index/` is git-ignored, so the pool index is rebuilt each session. The register
centroid is committed and already present.

In [ ]:
!python3 manage.py build_index --config configs/base_qwen.yaml

---
## Step 1 — the rarity list

In [ ]:
!python3 manage.py rarity --config configs/sparse_retrieval.yaml

In [ ]:
import json

import pandas as pd

rarity = json.load(open('results/rarity_train.json'))
cfg = rarity['config']
print(f"{rarity['n_terms']} pool terms, {rarity['n_eligible']} at df >= {cfg['min_df']} "
      f"-> {rarity['n_frozen']} frozen (requested {cfg['freeze_n']})")
print(f"realized df {rarity['df_observed']}, {rarity['selected_frac']:.1%} of the vocabulary")
print('pool df histogram  :', rarity['df_histogram']['pool'])
print('frozen df histogram:', rarity['df_histogram']['frozen'])

# The list is the 500 rarest above the floor, so it stops wherever the 500th term sits:
# a ceiling on df is an outcome of the freeze, not a parameter.
print(f"n_eligible / freeze_n = {rarity['n_eligible'] / cfg['freeze_n']:.2f}; "
      f"at 1.0 the floor alone decides the list and the ranking selects nothing")

In [ ]:
# Terms tie in df in bulk, and ties fall to total frequency and then to token order.
# If the list concentrates on a few first letters, the token order is choosing it.
initials = pd.Series([t[0] for t, *_ in rarity['terms']]).value_counts()
print(f'{len(initials)} distinct first letters over {rarity["n_frozen"]} terms; '
      f'the top 5 hold {initials.head(5).sum() / rarity["n_frozen"]:.1%}')
initials.head(12).to_frame('terms').T

In [ ]:
# The check that matters: these should read as marked vocabulary. If the head is broken
# segmentation instead, the df count is ranking tokenizer debris — stop and fix it.
top = pd.read_csv('results/rarity_train_top50.tsv', sep='\t')
top['example'] = top['example'].str.slice(0, 60)
top

### Normalization check — ZWNJ

In [ ]:
collisions = json.load(open('results/rarity_train.json'))['zwnj_collisions']
print(f'{len(collisions)} ZWNJ variant collisions')
pd.DataFrame(collisions, columns=['split spelling', 'joined spelling']).head(25)

---
## Step 2 — leakage audit

In [ ]:
!python manage.py leakage \
    --config configs/sparse_retrieval.yaml \
    --split val \
    --write-quarantine

In [ ]:
leak = json.load(open('results/leakage_val.json'))
print(f"val: {leak['n_eval_rows_flagged']}/{leak['n_eval_rows']} eval rows flagged, "
      f"{leak['n_pool_rows_flagged']} pool rows implicated")
print('  max-cos histogram:', leak['max_cos_histogram'])

pd.DataFrame([
    {'cos': f['cos'], 'jac_src': f['jaccard_source'], 'jac_tgt': f['jaccard_target'],
     'eval': f['eval_source'][:50], 'pool': f['pool_source'][:50]}
    for f in leak['flags'][:10]
])

In [ ]:
!python3 manage.py build_index --config configs/base_qwen.yaml \
    --index_dir data/knn_index_clean \
    --quarantine data/splits/pool_quarantine.json

---
## Step 3 — the sparse channel

The go/no-go. A query routes to the rare channel as soon as it carries one listed word,
so the numbers below are properties of the list: how many listed words a val query
carries, how often it carries any, and how often it carries four.

In [ ]:
!python3 manage.py sparse_select --config configs/sparse_retrieval.yaml \
    --split val --index_dir data/knn_index_clean

In [ ]:
sel = json.load(open('results/sparse_selection_val.json'))
print('routes            :', sel['route_fractions'])
print('rare words/query — mean', sel['query_terms']['mean'],
      'deciles', sel['query_terms']['deciles'])
print('share at or above :', sel['query_terms']['share_at_or_above'])
print('targeted served   :', sel['terms_served'])
print('rare slots filled :', sel['n_sparse']['histogram'])
print('intra-set cosine  :', sel['intra_set_similarity'])

In [ ]:
# One exemplar per selected word, so a query carrying four listed words fills four slots:
# the full-dose rate should track the share carrying four, unlike under greedy coverage.
EXPECTED = {'mean matches/query': 3.0, '>=1 match': 0.88, 'full dose (4 slots)': 0.34}
FULL_DOSE_GATE = 0.30

now = {
    'mean matches/query': sel['query_terms']['mean'],
    '>=1 match': sel['query_terms']['share_at_or_above']['1'],
    'full dose (4 slots)': sel['route_fractions']['full'],
}
print(pd.DataFrame({'now': now, 'expected': EXPECTED}).to_string())

gap = sel['query_terms']['share_at_or_above']['4'] - now['full dose (4 slots)']
print(f"\nqueries carrying 4+ listed words that did not fill 4 slots: {gap:.1%} "
      f"(every pool example carrying the word was already taken)")

full = now['full dose (4 slots)']
verdict = 'PROCEED' if full >= FULL_DOSE_GATE else 'HOLD'
print(f'full dose {full:.1%} against a {FULL_DOSE_GATE:.0%} gate -> {verdict}')
if verdict == 'HOLD':
    print('Report these numbers; do not generate on this list.')

In [ ]:
import numpy as np
import yaml

from src.retrieval.rarity import load_irregular
from src.retrieval.retrieve import RetrievalIndex
from src.retrieval.sparse import SparseRetriever

# Rare picks ran 15.4% shorter than the baseline's exemplars on the old list. A rare arm
# whose prompts are systematically shorter confounds quality with prompt length.
CFG = yaml.safe_load(open('configs/sparse_retrieval.yaml'))
RETR, SPA, RAR = CFG['retrieval'], CFG['sparse'], CFG['rarity']
SRC = [json.loads(ln)['input'] for ln in open('data/splits/val.jsonl') if ln.strip()]

index = RetrievalIndex('data/knn_index_clean', embed_model=RETR['embed_model'])
retriever = SparseRetriever(
    index, load_irregular(RAR['out']), index, zwnj=RAR['zwnj'], m=SPA['m'],
)
SELECTED, TRACES = retriever.select_with_trace(SRC, k=RETR['k'])
BASE_SEL = index.retrieve(SRC, k=RETR['k'])

RAR_LEN, COS_LEN = [], []
for t in TRACES:
    rows = set(t['sparse_rows'])
    for r in t['final_rows']:
        (RAR_LEN if r in rows else COS_LEN).append(len(index.pairs[r]['input']))

BASE_CHARS = sum(len(e['input']) for row in BASE_SEL for e in row) / len(TRACES)
ARM_CHARS = (sum(RAR_LEN) + sum(COS_LEN)) / len(TRACES)
for name, v in (('query', [len(x) for x in SRC]), ('rare channel', RAR_LEN),
                ('cosine fill', COS_LEN)):
    a = np.array(v)
    print(f'{name:14s} n={len(a):6d}  mean {a.mean():6.1f}  median {np.median(a):6.1f} chars')
print(f'exemplar chars per prompt: dense {BASE_CHARS:.0f} -> sparse {ARM_CHARS:.0f} '
      f'({ARM_CHARS / BASE_CHARS - 1:+.1%})')

In [ ]:
for min_df in (1, 10, 30, 60):
    lst = f'results/rarity_train_mindf{min_df}.json'
    print(f'--- min_df={min_df}')
    !python3 manage.py rarity --config configs/sparse_retrieval.yaml --min_df {min_df} --out {lst} 2>&1 | grep -E 'frozen|realized'
    !python3 manage.py sparse_select --config configs/sparse_retrieval.yaml --split val --index_dir data/knn_index_clean --rarity {lst} --out results/sparse_sweep_val_mindf{min_df}.json 2>&1 | grep -E 'routes|rare slots'

In [ ]:
for ex in sel['examples']:
    print('QUERY :', ex['source'][:90])
    print('  route', ex['trace']['route'], '| carries', ex['trace']['query_terms'])
    print('  served', ex['trace']['served_terms'], f"+ {ex['trace']['n_knn']} kNN")
    for e in ex['exemplars']:
        print('   -', e[:90])
    print()